# معمل البث اللحظي للتوكنز والمعالجة المتوازية (`stream` & `batch`)

أهلاً بك في المعمل العملي للمحاضرة 60!

في المعمل ده هنجرب الفرق الحقيقي بين 3 طرق لتشغيل الموديل:
1. **`invoke`**: بتستنى الموديل يخلص كتابة الرد كله في صمت، وبعدين يطبع كل الكلام مرة واحدة.
2. **`stream`**: الموديل بيبعت الكلمات جزء بجزء أول ما بتتولد قدام عينيك (زي ChatGPT).
3. **`batch`**: بنبعت 3 أسئلة مختلفة في نفس اللحظة بالتوازي، مع تحديد حد أقصى للتزامن (`max_concurrency`).


In [1]:
# تجهيز الموديل وأدوات قياس الوقت
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("تم تحميل الموديل وجاهزين نختبر سرعة التنفيذ!")


تم تحميل الموديل وجاهزين نختبر سرعة التنفيذ!


### 1. تجربة الطريقة العادية `model.invoke()`

لاحظ هنا إننا بنستنى لحد ما كل الجملة تخلص على السيرفر وبعدين بتظهر كلها مرة واحدة.


In [2]:
t0 = time.perf_counter()
res = model.invoke("Why do parrots have colorful feathers? Answer in 2 short sentences.")
print(f"استغرق الرد الكامل: {time.perf_counter() - t0:.2f} ثانية")
print("الإجابة:", res.content)


استغرق الرد الكامل: 1.31 ثانية
الإجابة: Parrots use their bright plumage to attract mates and signal good health. The colors also help them blend into the dappled light of their forest habitats.


### 2. تجربة البث اللحظي `model.stream()`

هنا هنشوف إزاي `stream()` بيرجع مُولّد (`Generator`). هنستخدم `chunk.text` ومعامل `end="|"` (عشان تشوف التقطيع بعينيك) ومعامل `flush=True` عشان يطبع فوراً.


In [3]:
print("البث اللحظي (الفاصل | يوضح وصول كل Chunk):")
chunks_count = 0

for chunk in model.stream("Why do parrots have colorful feathers? Answer in 2 short sentences."):
    print(chunk.text, end="|", flush=True)
    chunks_count += 1

print(f"\n\nعدد الأجزاء المستلمة أثناء البث: {chunks_count}")


البث اللحظي (الفاصل | يوضح وصول كل Chunk):
|

|

|

|

|

|

... (165 outputs left out)

.|

|

|



عدد الأجزاء المستلمة أثناء البث: 173


### 3. تجربة المعالجة المتوازية `model.batch()` مع `max_concurrency`

تخيل عندك 3 أسئلة مستقلة وعايز إجاباتهم كلهم بسرعة بدل ما تسأل واحد ورا التاني. تعال نبعتهم مع بعض بحد أقصى 3 اتصالات متوازية.


In [4]:
questions = [
    "Why do parrots have colorful feathers? 1 sentence.",
    "How do airplanes fly? 1 sentence.",
    "What is quantum computing? 1 sentence."
]

t0 = time.perf_counter()
answers = model.batch(questions, config={"max_concurrency": 3})
print(f"تمت الإجابة على الأسئلة الثلاثة معاً في {time.perf_counter() - t0:.2f} ثانية:\n")

for i, ans in enumerate(answers, 1):
    print(f"[{i}] {ans.content}")


تمت الإجابة على الأسئلة الثلاثة معاً في 0.93 ثانية:

[1] Parrots have colorful feathers mainly to attract mates and communicate with their flock, with pigments and structural coloration producing vivid hues that also help them blend into their forest habitats.
[2] Airplanes fly because their wings, shaped to accelerate air downward, create a pressure difference that generates lift, while engines provide thrust to overcome drag and keep the aircraft moving forward.
[3] Quantum computing is a computational paradigm that uses quantum bits (qubits) and principles like superposition, entanglement, and interference to perform calculations that can be exponentially faster for certain problems than classical computers.


### 4. جرب بنفسك!

دلوقتي دورك للتجربة الحرة:
1. جرب تغير الفاصل في البث اللحظي من `end="|"` إلى `end=""` وشوف الكلام بيظهر بانسيابية طبيعية زي الشات.
2. جرب تغير الأسئلة في `batch` أو تحط 5 أسئلة وتضبط `max_concurrency=2` وتلاحظ تأثير التوازي!


In [5]:
# جرب بنفسك هنا: بث ناعم بدون فواصل
my_prompt = "اكتب نصيحة برمجية ملهمة في سطرين باللغة العربية."

print("جاري البث:")
for chunk in model.stream(my_prompt):
    print(chunk.text, end="", flush=True)
print()


جاري البث:


ا

كتب

 ك

ود

ًا

... (28 outputs left out)

 أ

بد

ًا

.